# RAG tizimi: Supabase (RDB + VectorDB + GraphDB) + Router

**Bitta fayl, hammasi shu yerda.** Nima qiladi:
- PDF/TXT -> **VectorDB**, CSV -> **RDB**, bilim grafi -> **GraphDB** (hammasi Supabase'da)
- Savol berilganda **Router** (LLM) savolga mos bazani tanlaydi, shu bazadan ma'lumot olinadi, LLM javob yozadi

## Qanday ishlatiladi (3 qadam)
1. Yuqoridagi menyudan **Runtime -> Run all** (hammasini ishga tushirish) ni bosing.
2. Kalitlarni so'raganda kiriting (kiritilgan kalitlar fayl ichida **saqlanmaydi**).
3. Agar notebook "SQL kerak" desa: u o'zi SQL matnini va havolani ko'rsatadi. SQL ni Supabase'da bir marta Run qiling, so'ng **Runtime -> Run all** ni qayta bosing.

## 1. Kutubxonalarni o'rnatish (1-3 daqiqa)

In [ ]:
!pip install -q langchain-core langchain-community langchain-text-splitters langchain-google-genai langchain-groq langchain-huggingface sentence-transformers supabase pypdf reportlab pandas pydantic

## 2. Kalitlarni kiritish
Savollar ketma-ket chiqadi. Kalit yozilganda ko'rinmaydi, bu normal. Supabase URL oxiridagi `/rest/v1/` bo'lsa ham xato bo'lmaydi.

In [ ]:
import os, re
from getpass import getpass

raw_url = input("1) Supabase Project URL (https://....supabase.co): ").strip().strip("'\"")
m = re.search(r"https://[a-z0-9]+\.supabase\.co", raw_url)
if not m:
    raise ValueError("URL noto'g'ri. Misol: https://abcdefgh.supabase.co")
SUPABASE_URL = m.group(0)

SUPABASE_KEY = getpass("2) Supabase service_role (yoki secret) kaliti: ").strip().strip("'\"")
if not SUPABASE_KEY:
    raise ValueError("Supabase kaliti bo'sh.")

for env_name, prompt in [
    ("GOOGLE_API_KEY", "3) Gemini API kaliti (bo'lmasa Enter): "),
    ("GROQ_API_KEY", "4) Groq API kaliti (bo'lmasa Enter): "),
]:
    value = getpass(prompt).strip().strip("'\"")
    if value:
        os.environ[env_name] = value
    else:
        os.environ.pop(env_name, None)

if not os.getenv("GOOGLE_API_KEY") and not os.getenv("GROQ_API_KEY"):
    raise ValueError("Kamida bitta LLM kaliti (Gemini yoki Groq) kerak.")
print("Kalitlar qabul qilindi. Supabase:", SUPABASE_URL)

## 3. Supabase'ga ulanish va jadvallarni tekshirish
Jadvallar yo'q bo'lsa, bu katakcha **SQL matnini va havolani** chiqaradi va to'xtaydi. Shunda: havolani oching, SQL ni qo'ying, **Run** bosing, so'ng Runtime -> Run all.

In [ ]:
from supabase import create_client

SETUP_SQL = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS structured_data (
    id SERIAL PRIMARY KEY,
    entity_name TEXT,
    category TEXT,
    metric_value NUMERIC,
    metadata JSONB
);

CREATE TABLE IF NOT EXISTS document_vectors (
    id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    content TEXT,
    metadata JSONB,
    embedding VECTOR(384)
);

CREATE OR REPLACE FUNCTION match_documents (
    query_embedding VECTOR(384),
    match_threshold FLOAT DEFAULT -1.0,
    match_count INT DEFAULT 5
)
RETURNS TABLE (id UUID, content TEXT, metadata JSONB, similarity FLOAT)
LANGUAGE sql STABLE
AS $$
    SELECT dv.id, dv.content, dv.metadata,
           (1 - (dv.embedding <=> query_embedding))::float AS similarity
    FROM document_vectors AS dv
    WHERE 1 - (dv.embedding <=> query_embedding) > match_threshold
    ORDER BY dv.embedding <=> query_embedding
    LIMIT match_count;
$$;

CREATE TABLE IF NOT EXISTS graph_nodes (
    node_id TEXT PRIMARY KEY,
    label TEXT,
    properties JSONB
);

CREATE TABLE IF NOT EXISTS graph_edges (
    id SERIAL PRIMARY KEY,
    source_id TEXT REFERENCES graph_nodes(node_id),
    target_id TEXT REFERENCES graph_nodes(node_id),
    relation TEXT,
    properties JSONB,
    UNIQUE (source_id, target_id, relation)
);

NOTIFY pgrst, 'reload schema';
"""

client = create_client(SUPABASE_URL, SUPABASE_KEY)
PROJECT_REF = SUPABASE_URL.split("//")[1].split(".")[0]
TABLES = ["structured_data", "document_vectors", "graph_nodes", "graph_edges"]

missing, other_error = [], None
for t in TABLES:
    try:
        client.table(t).select("*").limit(1).execute()
    except Exception as e:
        msg = str(e)
        if "PGRST205" in msg or "does not exist" in msg or "schema cache" in msg:
            missing.append(t)
        else:
            other_error = msg
            break

if other_error:
    raise RuntimeError(
        "Supabase'ga ulanib bo'lmadi. URL yoki kalitni tekshiring (2-katakchani qayta ishga tushiring).\n"
        f"Xato: {other_error}"
    )

if missing:
    print("Jadvallar hali yaratilmagan:", ", ".join(missing))
    print()
    print("1) Shu havolani oching:")
    print(f"   https://supabase.com/dashboard/project/{PROJECT_REF}/sql/new")
    print("2) Pastdagi SQL ni to'liq nusxalab, oq maydonga qo'ying va yashil RUN tugmasini bosing.")
    print("3) 'Success' chiqqach, bu yerga qaytib Runtime -> Run all ni bosing.")
    print("\n" + "=" * 20 + " SQL BOSHLANDI " + "=" * 20)
    print(SETUP_SQL)
    print("=" * 20 + " SQL TUGADI " + "=" * 23)
    raise RuntimeError("SQL ni ishga tushiring, so'ng Runtime -> Run all ni qayta bosing.")

print("Supabase ulandi, 4 ta jadval tayyor.")

## 4. Embedding modeli va LLM'lar
Embedding: ko'p tilli (koreys, o'zbek, ingliz) 384 o'lchamli model. Birinchi marta model yuklanishi bir necha daqiqa olishi mumkin.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
_dim = len(embeddings.embed_query("test"))
if _dim != 384:
    raise RuntimeError(f"Embedding o'lchami {_dim}, jadval esa VECTOR(384) uchun. 384 o'lchamli model tanlang.")
print("Embedding tayyor, o'lcham:", _dim)

GEMINI_CANDIDATES = ["gemini-3.5-flash", "gemini-2.5-flash", "gemini-flash-latest"]
GROQ_MODEL = "openai/gpt-oss-20b"
LLMS = {}

if os.getenv("GOOGLE_API_KEY"):
    from langchain_google_genai import ChatGoogleGenerativeAI
    _last = None
    for _name in GEMINI_CANDIDATES:
        try:
            _m = ChatGoogleGenerativeAI(model=_name)
            _m.invoke("ping")
            LLMS["gemini"] = _m
            print("Gemini modeli ishlayapti:", _name)
            break
        except Exception as e:
            _last = e
    if "gemini" not in LLMS:
        print("OGOHLANTIRISH: Gemini ishlamadi:", _last)

if os.getenv("GROQ_API_KEY"):
    from langchain_groq import ChatGroq
    try:
        _g = ChatGroq(model=GROQ_MODEL, temperature=0.1)
        _g.invoke("ping")
        LLMS["groq"] = _g
        print("Groq modeli ishlayapti:", GROQ_MODEL)
    except Exception as e:
        print("OGOHLANTIRISH: Groq ishlamadi:", e)

if not LLMS:
    raise RuntimeError("Hech bir LLM ishlamadi. Kalitlarni tekshiring (2-katakchani qayta ishga tushiring).")

## 5. Namunaviy fayllarni yaratish (CSV, TXT, PDF)

In [ ]:
import pandas as pd
from pathlib import Path
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

DATA_DIR = Path("rag_data")
DATA_DIR.mkdir(exist_ok=True)

pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Project Alpha", "Project Beta", "Project Gamma"],
    "category": ["AI", "Cloud", "Data"],
    "value": [15000.0, 23000.0, 8500.0],
}).to_csv(DATA_DIR / "sample.csv", index=False, encoding="utf-8")

(DATA_DIR / "sample.txt").write_text(
    "Supabase는 오픈소스 Firebase 대안 플랫폼입니다.\n"
    "PostgreSQL을 기반으로 구축되었으며 pgvector 확장을 통해 벡터 임베딩 저장 및 유사도 검색을 지원합니다.\n"
    "LLM 어플리케이션 개발 시 VectorDB와 RDB 역할을 동시에 수행할 수 있어 매우 유용합니다.",
    encoding="utf-8",
)

c = canvas.Canvas(str(DATA_DIR / "sample.pdf"), pagesize=letter)
c.drawString(100, 750, "Project Alpha Overview Document")
c.drawString(100, 730, "Objective: Build a Next-Gen RAG Architecture.")
c.drawString(100, 710, "Technologies: LangChain, Supabase, Hugging Face, Gemini, ChatGroq.")
c.save()

print("Yaratildi:", sorted(p.name for p in DATA_DIR.iterdir()))

## 6. Indekslash: fayllarni 3 ta bazaga yozish
Har safar ishga tushganda avval eski ma'lumot tozalanadi, shuning uchun takroriy yozuvlar bo'lmaydi.

In [ ]:
import json
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

BATCH = 50

def reset_tables():
    client.table("graph_edges").delete().neq("id", 0).execute()
    client.table("graph_nodes").delete().neq("node_id", "").execute()
    client.table("structured_data").delete().neq("id", 0).execute()
    client.table("document_vectors").delete().neq("id", "00000000-0000-0000-0000-000000000000").execute()
    print("Eski ma'lumotlar tozalandi.")

def index_documents(paths):
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
    chunks = []
    for p in paths:
        if p.suffix.lower() == ".pdf":
            loader = PyPDFLoader(str(p))
        elif p.suffix.lower() == ".txt":
            loader = TextLoader(str(p), encoding="utf-8")
        else:
            continue
        chunks.extend(splitter.split_documents(loader.load()))
    chunks = [c for c in chunks if c.page_content.strip()]
    texts = [c.page_content.replace("\x00", "") for c in chunks]
    vectors = embeddings.embed_documents(texts)
    rows = [
        {"content": t, "metadata": json.loads(json.dumps(c.metadata, default=str)), "embedding": v}
        for t, c, v in zip(texts, chunks, vectors)
    ]
    for i in range(0, len(rows), BATCH):
        client.table("document_vectors").insert(rows[i:i + BATCH]).execute()
    print(f"[VectorDB] {len(rows)} ta chunk saqlandi")

def index_csv(path):
    df = pd.read_csv(path)
    records = json.loads(df.to_json(orient="records"))
    rows = []
    for rec in records:
        val = rec.get("value")
        rows.append({
            "entity_name": str(rec["name"]),
            "category": str(rec["category"]),
            "metric_value": float(val) if val is not None else None,
            "metadata": rec,
        })
    client.table("structured_data").insert(rows).execute()
    print(f"[RDB] {len(rows)} ta qator saqlandi")

TRIPLES = [  # (manba, manba turi, munosabat, maqsad, maqsad turi)
    ("Project Alpha", "Project", "USES_TECH", "LangChain", "Technology"),
    ("Project Alpha", "Project", "STORED_IN", "Supabase", "Technology"),
    ("Supabase", "Technology", "HAS_EXTENSION", "pgvector", "Technology"),
]

def index_graph(triples):
    nodes = {}
    for s, st, _r, t, tt in triples:
        nodes[s] = {"node_id": s, "label": st}
        nodes.setdefault(t, {"node_id": t, "label": tt})
    client.table("graph_nodes").upsert(list(nodes.values())).execute()
    edges = [{"source_id": s, "target_id": t, "relation": r} for s, _st, r, t, _tt in triples]
    client.table("graph_edges").upsert(edges, on_conflict="source_id,target_id,relation").execute()
    print(f"[GraphDB] {len(nodes)} ta tugun, {len(edges)} ta munosabat saqlandi")

reset_tables()
index_documents([DATA_DIR / "sample.txt", DATA_DIR / "sample.pdf"])
index_csv(DATA_DIR / "sample.csv")
index_graph(TRIPLES)
print("\nIndekslash tugadi. Supabase > Table Editor da ma'lumotlarni ko'rishingiz mumkin.")

## 7. Router + Retrieval + Generation

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    destination: Literal["rdb", "vectordb", "graphdb"] = Field(
        ...,
        description=(
            "Select DB type based on the query: 'rdb' for numeric/tabular/stats, "
            "'vectordb' for textual details/unstructured info, "
            "'graphdb' for relationship/network/connection queries."
        ),
    )

def to_text(content):
    """LLM javobini (matn yoki bo'laklar ro'yxati) oddiy matnga aylantiradi."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for p in content:
            if isinstance(p, str):
                parts.append(p)
            elif isinstance(p, dict) and p.get("type") == "text":
                parts.append(p.get("text", ""))
        return "".join(parts)
    return str(content)

def pick_llm(model_type):
    if model_type in LLMS:
        return LLMS[model_type]
    other = next(iter(LLMS))
    print(f"OGOHLANTIRISH: '{model_type}' mavjud emas, '{other}' ishlatiladi.")
    return LLMS[other]

def route_query(llm, query):
    try:
        result = llm.with_structured_output(RouteQuery).invoke(
            f"Analyze the user query and pick the best database destination:\nQuery: {query}"
        )
        return result.destination
    except Exception as e:
        print(f"OGOHLANTIRISH: router ishlamadi ({e}); 'vectordb' tanlandi.")
        return "vectordb"

def retrieve(target, query):
    if target == "vectordb":
        rows = client.rpc("match_documents", {
            "query_embedding": embeddings.embed_query(query),
            "match_threshold": -1.0,
            "match_count": 3,
        }).execute().data
        if not rows:
            return "(no matching documents found)"
        return "\n---\n".join(r["content"] for r in rows)

    if target == "rdb":
        rows = client.table("structured_data").select("*").limit(100).execute().data
        if not rows:
            return "(table structured_data is empty)"
        total = sum(float(r["metric_value"]) for r in rows if r.get("metric_value") is not None)
        lines = [f"- {r['entity_name']} | category={r['category']} | value={r['metric_value']}" for r in rows]
        return "\n".join(lines) + f"\nRow count: {len(rows)}\nSum of value (precomputed): {total}"

    nodes = client.table("graph_nodes").select("*").execute().data
    edges = client.table("graph_edges").select("*").execute().data
    if not nodes and not edges:
        return "(graph tables are empty)"
    return ("Nodes:\n" + "\n".join(f"- {n['node_id']} ({n.get('label')})" for n in nodes)
            + "\nEdges:\n" + "\n".join(f"- {e['source_id']} -[{e['relation']}]-> {e['target_id']}" for e in edges))

def run_pipeline(user_query, model_type="gemini"):
    llm = pick_llm(model_type)
    target = route_query(llm, user_query)
    print(f"[Router] {target.upper()}  <-  {user_query}")
    context = retrieve(target, user_query)
    prompt = f"""You are a knowledge assistant. Answer the question based ONLY on the retrieved data context provided below.
If the answer cannot be determined from the context, state that you don't have enough information.
Answer in the same language as the user's question.

[Retrieved Database Source]: {target.upper()}
[Retrieved Context]:
{context}

[User Question]:
{user_query}

[Answer]:"""
    answer = to_text(llm.invoke(prompt).content)
    print("================ [ Final Answer ] ================")
    print(answer)
    print("==================================================\n")
    return answer

## 8. Testlar (topshiriqdagi 3 ta savol)
Kutilgan: 1-savol -> VECTORDB, 2-savol -> RDB (jami **46500**), 3-savol -> GRAPHDB.

In [ ]:
print("--- Test 1: Hujjat (VectorDB) ---")
run_pipeline("Supabase가 LLM 애플리케이션 개발에서 어떤 장점이 있는지 문서 내용을 바탕으로 설명해줘.", "groq")

print("--- Test 2: Jadval (RDB) ---")
run_pipeline("현재 등록되어 있는 프로젝트들의 이름, 카테고리, value 값을 확인하고 전체 value 합계를 계산해줘.", "gemini")

print("--- Test 3: Munosabat (GraphDB) ---")
run_pipeline("Project Alpha는 어떤 기술(Technology)들과 연결되어 있고, Supabase는 어떤 확장 기능을 가지고 있는지 관계를 알려줘.", "groq")

## 9. O'z savolingizni bering
Pastdagi `QUESTION` matnini o'zgartirib, katakchani ishga tushiring.

In [ ]:
QUESTION = "Project Beta의 value는 얼마야?"
run_pipeline(QUESTION, "gemini")